# Architecture Recovery for the Zeeguu API Application

This notebook is my attempt at a static architecture recovery of the Zeeguu API by parsing Python imports with `ast`, building dependency graphs with `networkx`, and exporting interactive visualizations with `pyvis`.

An edge `A → B` means that module or package `A` imports `B`.

## 1. Imports and Configuration

Set `CODE_ROOT_FOLDER` to the local path of the Zeeguu API repository you want to analyse.

In [15]:
import os
import ast
import warnings
from pathlib import Path

import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame, display

# Suppress SyntaxWarning noise from parsed files
warnings.filterwarnings("ignore", category=SyntaxWarning)

# Print current working directory if needed
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# Set path to the repository you are doing architecture recovery on.
CODE_ROOT_FOLDER = "../zeeguu-api/"
print(f"Code root folder: {CODE_ROOT_FOLDER}")

Current working directory: /mnt/d/GitHub/software-architecture-individual
Code root folder: ../zeeguu-api/


## 2. Path and Module Helper Functions

In [16]:
def file_path(file_name: str) -> str:
  """
  Get the full path of a file in the codebase from its relative path.
  """
  return CODE_ROOT_FOLDER + file_name


def module_name_from_file_path(full_path: str) -> str:
  """
  Convert a Python file path to a Python module name.

  Example:
      ../zeeguu-api/zeeguu/core/model/user.py
      -> zeeguu.core.model.user
  """
  file_name = full_path[len(CODE_ROOT_FOLDER):]
  file_name = file_name.replace("/__init__.py", "")
  file_name = file_name.replace("/", ".")
  file_name = file_name.replace(".py", "")
  return file_name


def in_package(module_name: str, package_prefix: str) -> bool:
  """
  Check whether a module belongs to a package.
  """
  return module_name == package_prefix or module_name.startswith(package_prefix + ".")


def package_of(module_name: str, depth: int = 3) -> str:
  """
  Convert a module name to a package name at a given depth.
  Depth set to 3 as default.

  Example:
      zeeguu.core.model.user -> zeeguu.core.model, with depth=3
  """
  return ".".join(module_name.split(".")[:depth])

## 3. Sanity Checks

These assertions verify that the path/module conversion helpers behave as expected.

In [17]:
assert file_path("zeeguu/core/model/user.py") == "../zeeguu-api/zeeguu/core/model/user.py"
assert module_name_from_file_path(file_path("zeeguu/core/model/user.py")) == "zeeguu.core.model.user"

print("Helper function sanity checks passed.")

Helper function sanity checks passed.


## 4. Extract Imports from Python Files

This function uses the Python `ast` module to parse imports without executing the target files.

In [18]:
def imports_from_file(file_path: str) -> list[str]:
  """
  Extract imported modules from a Python file using ast.

  Handles:
  - import x
  - import x, y
  - from x import y
  - from . import y, represented as "."
  """
  imports = []

  with open(file_path, encoding="utf-8", errors="ignore") as f:
    tree = ast.parse(f.read(), filename=file_path)

  for node in ast.walk(tree):
    # import x, y, z
    if isinstance(node, ast.Import):
      for name in node.names:
        imports.append(name.name)

    # from x import y, z
    elif isinstance(node, ast.ImportFrom):
      if node.module is not None:
        imports.append(node.module)
      else:
        # handles: from . import y
        imports.append(".")

  return imports

## 5. Inspect Imports from Example Files

As a sanity check I list the imports from three files from the model in Zeeguu API, namely: user.py, bookmark.py, and unique_code.py

In [19]:
user_imports = imports_from_file(file_path("zeeguu/core/model/user.py"))
bookmark_imports = imports_from_file(file_path("zeeguu/core/model/bookmark.py"))
unique_code_imports = imports_from_file(file_path("zeeguu/core/model/unique_code.py"))

print("Imports from user.py:")
print(user_imports)

print("\nImports from bookmark.py:")
print(bookmark_imports)

print("\nImports from unique_code.py:")
print(unique_code_imports)

assert unique_code_imports != bookmark_imports
print("\nImport extraction sanity check passed.")

Imports from user.py:
['datetime', 'json', 'random', 're', 'sqlalchemy.orm', 'sqlalchemy', 'sqlalchemy.orm', 'sqlalchemy.orm.exc', 'zeeguu.core', 'zeeguu.core.language.difficulty_estimator_factory', 'zeeguu.core.model.db', 'zeeguu.core.model.language', 'zeeguu.core.util.hash', 'zeeguu.logging', 'zeeguu.logging', 'datetime', 'zeeguu.core.model.user_cohort_map', 'datetime', 'zeeguu.core.model', 'zeeguu.core.model.bookmark', 'zeeguu.core.model.user_avatar', 'zeeguu.core.model.user_word', 'zeeguu.core.model.user_preference', 'zeeguu.core.model', 'zeeguu.core.model', 'zeeguu.core.model.user_language', 'zeeguu.core.word_scheduling.basicSR.basicSR', 'zeeguu.core.sql.queries.query_loader', 'zeeguu.core.sql.query_building', 'zeeguu.core.model.bookmark', 'zeeguu.core.model.user_word', 'zeeguu.core.model.exercise', 'zeeguu.core.model.phrase', 'zeeguu.core.model.meaning', 'zeeguu.core.model.user_word', 'zeeguu.core.model.exercise', 'zeeguu.core.model.phrase', 'zeeguu.core.model.meaning', 'zeeguu.c

## 6. Graph Drawing

The graph is saved as an interactive HTML file under `./graphs/`. In a notebook, the generated HTML is also displayed inline using an `IFrame`.

In [20]:
def draw_graph(G: nx.DiGraph, figure_name: str, height: str = "900px", width: str = "100%", notebook: bool = True):
  """
  Draw a directed graph using PyVis and save it as an interactive HTML file.

  Edge A -> B means A imports B.
  """
  os.makedirs("./graphs", exist_ok=True)

  net = Network(
    height=height,
    width=width,
    directed=True,
    notebook=notebook,
    bgcolor="#ffffff",
    font_color="#000000",
  )

  # Optional physics layout settings
  net.barnes_hut(
    gravity=-30000,
    central_gravity=0.3,
    spring_length=200,
    spring_strength=0.05,
    damping=0.09,
    overlap=0,
  )

  # Add nodes
  for node in G.nodes:
    net.add_node(node, label=node, title=node)

  # Add edges
  for source, target in G.edges:
    net.add_edge(source, target, title=f"{source} imports {target}")

  # Add interaction buttons
  net.show_buttons(filter_=["physics"])

  output_path = f"./graphs/{figure_name}.html"
  net.write_html(output_path)

  print(f"Saved interactive graph to: {output_path}")
  # display(IFrame(src=output_path, width="100%", height=height.replace("px", "")))

  return output_path

## 7. Build Module-Level Dependency Graph

This graph keeps dependencies at full module granularity.

In [ ]:
def dependencies_digraph(code_root_folder: str) -> nx.DiGraph:
  """
  Build a directed graph of module dependencies.

  Edge A -> B means module A imports module B.
  """
  files = Path(code_root_folder).rglob("*.py")
  G = nx.DiGraph()

  for file in files:
    source_file_path = str(file)
    source_module = module_name_from_file_path(source_file_path)

    if source_module not in G.nodes:
      G.add_node(source_module)

    for target_module in imports_from_file(source_file_path):
      if target_module.startswith("zeeguu"):
        G.add_edge(source_module, target_module)

  return G

## 8. Build Package-Level Dependency Graph

This graph groups modules into packages according to the chosen depth.

Examples:

- `depth=2`: `zeeguu.core`
- `depth=3`: `zeeguu.core.model`
- `depth=4`: `zeeguu.core.model.user`

In [ ]:
def package_dependencies_digraph(code_root_folder: str, depth: int = 3) -> nx.DiGraph:
  """
  Build a directed graph of package dependencies.

  Example:
    zeeguu.core.model.user imports zeeguu.core.model.bookmark
    becomes zeeguu.core.model -> zeeguu.core.model when depth=3.

  Self-dependencies are omitted.
  """
  files = Path(code_root_folder).rglob("*.py")
  G = nx.DiGraph()

  for file in files:
    source_module = module_name_from_file_path(str(file))

    if not source_module.startswith("zeeguu"):
      continue

    source_pkg = package_of(source_module, depth)
    G.add_node(source_pkg)

    for target_module in imports_from_file(str(file)):
      if target_module.startswith("zeeguu"):
        target_pkg = package_of(target_module, depth)
        if source_pkg != target_pkg:
          G.add_edge(source_pkg, target_pkg)

  return G

## 9. Generate Architecture Graphs at Different Granularities

In [23]:
DG_3 = package_dependencies_digraph(CODE_ROOT_FOLDER, depth=3)
print(f"Depth 3 graph: {DG_3.number_of_nodes()} nodes, {DG_3.number_of_edges()} edges")
draw_graph(DG_3, "architecture-packages")

Depth 3 graph: 59 nodes, 192 edges
Saved interactive graph to: ./graphs/architecture-packages.html


'./graphs/architecture-packages.html'

In [24]:
DG_2 = package_dependencies_digraph(CODE_ROOT_FOLDER, depth=2)
print(f"Depth 2 graph: {DG_2.number_of_nodes()} nodes, {DG_2.number_of_edges()} edges")
draw_graph(DG_2, "architecture-packages-depth-2")

Depth 2 graph: 8 nodes, 13 edges
Saved interactive graph to: ./graphs/architecture-packages-depth-2.html


'./graphs/architecture-packages-depth-2.html'

In [25]:
DG_4 = package_dependencies_digraph(CODE_ROOT_FOLDER, depth=4)
print(f"Depth 4 graph: {DG_4.number_of_nodes()} nodes, {DG_4.number_of_edges()} edges")
draw_graph(DG_4, "architecture-packages-depth-4")

Depth 4 graph: 370 nodes, 1223 edges
Saved interactive graph to: ./graphs/architecture-packages-depth-4.html


'./graphs/architecture-packages-depth-4.html'

## 10. Subgraphs of Specific Packages

In [26]:
# Core package: main domain logic, including models, services, and repositories.
DG_core = DG_3.subgraph([n for n in DG_3.nodes if n.startswith("zeeguu.core")]).copy()
print(f"Core subgraph: {DG_core.number_of_nodes()} nodes, {DG_core.number_of_edges()} edges")
draw_graph(DG_core, "architecture-packages-core")

Core subgraph: 42 nodes, 110 edges
Saved interactive graph to: ./graphs/architecture-packages-core.html


'./graphs/architecture-packages-core.html'

In [27]:
DG_model = DG_4.subgraph([n for n in DG_4.nodes if n.startswith("zeeguu.core.model")]).copy()
print(f"Model subgraph: {DG_model.number_of_nodes()} nodes, {DG_model.number_of_edges()} edges")
draw_graph(DG_model, "architecture-packages-core-model")

Model subgraph: 104 nodes, 334 edges
Saved interactive graph to: ./graphs/architecture-packages-core-model.html


'./graphs/architecture-packages-core-model.html'

## 11. Basic Graph Metrics

These metrics can help you identify central packages/modules and possible architectural hotspots.

In [ ]:
def print_graph_summary(G: nx.DiGraph, name: str):
  print(f"{name}")
  print("-" * len(name))
  print(f"Nodes: {G.number_of_nodes()}")
  print(f"Edges: {G.number_of_edges()}")

  if G.number_of_nodes() > 0:
    in_degree = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:10]
    out_degree = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:10]

    print("\nTop imported packages/modules:")
    for node, degree in in_degree:
      print(f"  {node}: {degree}")

    print("\nTop importing packages/modules:")
    for node, degree in out_degree:
      print(f"  {node}: {degree}")

print_graph_summary(DG_3, "Package Dependency Graph, Depth 3")

Package Dependency Graph, Depth 3
---------------------------------
Nodes: 59
Edges: 192

Top imported packages/modules:
  zeeguu.core.model: 34
  zeeguu.logging: 19
  zeeguu.core: 14
  zeeguu: 8
  zeeguu.core.util: 8
  zeeguu.core.tokenization: 7
  zeeguu.core.word_scheduling: 7
  zeeguu.core.llm_services: 6
  zeeguu.config: 5
  zeeguu.core.constants: 5

Top importing packages/modules:
  zeeguu.api.endpoints: 30
  zeeguu.core.model: 23
  zeeguu.core.test: 14
  zeeguu.core.content_retriever: 10
  zeeguu.api.app: 9
  zeeguu.core.audio_lessons: 7
  zeeguu.operations.crawler: 6
  zeeguu.core.llm_services: 6
  zeeguu.core.util: 6
  zeeguu.core.semantic_search: 5
